# Rossmann Store Sales — Feature Engineering per RNN

Notebook di preparazione dati per un modello ricorrente (RNN) di forecasting delle vendite. Carichiamo train / validation / test, uniamo le informazioni dei negozi (`store.csv`) e applichiamo la pipeline di feature engineering richiesta: encoding ciclico di data/giorno della settimana, encoding delle variabili categoriche, gestione dei valori mancanti e trasformazione del target.

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "dataset"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)


In [ ]:
print("=" * 70)
print("1. CARICAMENTO DATI: train, validation, test + merge con store")
print("=" * 70)

# StateHoliday viene letta come stringa per evitare che pandas mescoli
# il valore 0 (int) con "a"/"b"/"c" (str) nella stessa colonna (colonna a
# tipo misto -> warning e comportamento inconsistente nelle operazioni
# successive, es. il confronto "!= '0'" usato piu' avanti).
train_raw = pd.read_csv(
    os.path.join(DATA_DIR, "train.csv"),
    parse_dates=["Date"], dtype={"StateHoliday": str}, low_memory=False,
)
test_raw = pd.read_csv(
    os.path.join(DATA_DIR, "test.csv"),
    parse_dates=["Date"], dtype={"StateHoliday": str}, low_memory=False,
)
store = pd.read_csv(os.path.join(DATA_DIR, "store.csv"))

print(f"train.csv: {train_raw.shape[0]:,} righe, {train_raw.shape[1]} colonne")
print(f"test.csv:  {test_raw.shape[0]:,} righe, {test_raw.shape[1]} colonne")
print(f"store.csv: {store.shape[0]:,} righe, {store.shape[1]} colonne")

# Merge con le informazioni sul negozio (StoreType, Assortment, concorrenza,
# Promo2, ...), disponibili sia per il train che per il test
train_full = train_raw.merge(store, how="left", on="Store")
test_df = test_raw.merge(store, how="left", on="Store")

print(f"\ntrain unito a store: {train_full.shape[0]:,} righe, {train_full.shape[1]} colonne")
print(f"test unito a store:  {test_df.shape[0]:,} righe, {test_df.shape[1]} colonne")


In [ ]:
print("\n" + "=" * 70)
print("2. SPLIT TEMPORALE: train / validation")
print("=" * 70)

# Non disponiamo di un file di validation separato: lo ricaviamo dal train,
# tenendo da parte le ultime settimane in ordine CRONOLOGICO (mai uno split
# casuale su una serie storica: mescolare le date causerebbe data leakage,
# facendo "vedere" al modello in training giorni futuri rispetto a quelli di
# validazione). La finestra di validation replica la durata del test
# ufficiale (~6 settimane), cosi' la metrica calcolata in validazione e'
# rappresentativa di quella attesa sul test.
VAL_WEEKS = 6

train_full = train_full.sort_values("Date").reset_index(drop=True)
cutoff_date = train_full["Date"].max() - pd.Timedelta(weeks=VAL_WEEKS)

train_df = train_full[train_full["Date"] <= cutoff_date].copy()
val_df = train_full[train_full["Date"] > cutoff_date].copy()

print(f"Train:      {len(train_df):>8,} righe  ({train_df['Date'].min().date()} -> {train_df['Date'].max().date()})")
print(f"Validation: {len(val_df):>8,} righe  ({val_df['Date'].min().date()} -> {val_df['Date'].max().date()})")
print(f"Test:       {len(test_df):>8,} righe  ({test_df['Date'].min().date()} -> {test_df['Date'].max().date()})")


In [ ]:
print("\n" + "=" * 70)
print("3. CORREZIONE VALORI MANCANTI NOTI NEL TEST UFFICIALE")
print("=" * 70)

# Bug noto del test set ufficiale Rossmann: la colonna Open ha alcuni valori
# mancanti per lo Store 622. Li impostiamo a 1 (negozio aperto): e' l'ipotesi
# piu' ragionevole, dato che negli altri giorni della stessa settimana quello
# store risulta regolarmente aperto, e comunque necessaria per non avere NaN
# in input alla rete.
n_missing_open = test_df["Open"].isnull().sum()
print(f"Valori mancanti in Open (test): {n_missing_open}")
test_df["Open"] = test_df["Open"].fillna(1).astype(int)


In [ ]:
print("\n" + "=" * 70)
print("4. STATISTICHE CALCOLATE SOLO SUL TRAINING SET")
print("=" * 70)

# La mediana di CompetitionDistance viene calcolata SOLO sul training set e
# poi riusata (senza ricalcolarla) per riempire i mancanti anche in
# validation e test: se la calcolassimo sull'intero dataset, faremmo
# trapelare nel training un'informazione statistica derivata anche da
# osservazioni future (data leakage).
competition_distance_median = train_df["CompetitionDistance"].median()
print(f"Mediana CompetitionDistance (dal training set): {competition_distance_median}")


## Pipeline di feature engineering

Definiamo un'unica funzione `engineer_features` e la applichiamo identica a train, validation e test: questo garantisce che le tre tabelle finiscano con le **stesse colonne, nello stesso ordine e con la stessa logica di trasformazione**, requisito indispensabile per alimentare correttamente una rete neurale. Le uniche informazioni "esterne" alla singola riga (la mediana di `CompetitionDistance`) vengono passate come parametro, calcolate una sola volta sul training set (cella precedente).

In [ ]:
def engineer_features(
    df,
    competition_distance_median,
    is_train,
    drop_store=True,
    dayofweek_sincos=True,
    date_sincos=True,
    drop_customers=True,
    stateholiday_bool=True,
    storetype_onehot=True,
    assortment_int=True,
    fill_competition_distance=True,
    competition_open_months=True,
    promo2_open_weeks=True,
    promo_interval_bool=True,
    sales_log=True,
):
    """
    Applica al dataframe (train, validation oppure test) le trasformazioni
    di feature engineering per il modello RNN. Ogni trasformazione puo'
    essere attivata/disattivata singolarmente tramite i parametri booleani,
    per poter sperimentare facilmente con sottoinsiemi diversi di feature.

    Parametri
    ---------
    df : dataframe grezzo (train/val gia' unito a store, oppure test unito
         a store)
    competition_distance_median : mediana di CompetitionDistance calcolata
         SOLO sul training set, riusata per train/val/test
    is_train : True per train/validation (contengono la colonna Sales),
         False per il test ufficiale (che non la contiene)
    drop_store : rimuove la colonna Store (identificativo negozio)
    dayofweek_sincos : trasforma DayOfWeek in DayOfWeek_sin/DayOfWeek_cos
         e rimuove la colonna originale
    date_sincos : ricava DayOfYear_sin/DayOfYear_cos (stagionalita' annuale,
         periodo 365 giorni) e rimuove la colonna Date
    drop_customers : rimuove la colonna Customers (non nota a tempo di
         predizione)
    stateholiday_bool : converte StateHoliday da stringa ("0","a","b","c")
         a booleano/intero 0/1 (0 = nessuna festivita')
    storetype_onehot : converte StoreType in one-hot encoding
    assortment_int : converte Assortment in intero (a=0, b=1, c=2)
    fill_competition_distance : riempie i valori mancanti di
         CompetitionDistance con la mediana passata come parametro
    competition_open_months : sostituisce CompetitionOpenSinceMonth/Year con
         CompetitionOpenMonths ("da quanti mesi e' aperta la concorrenza"),
         mancanti e negativi impostati a 0
    promo2_open_weeks : sostituisce Promo2SinceYear/Week con Promo2OpenWeeks
         ("da quante settimane e' attiva Promo2"), mancanti e negativi
         impostati a 0
    promo_interval_bool : sostituisce PromoInterval con il booleano
         IsPromoMonth (il mese corrente e' un mese di ripartenza di Promo2?)
    sales_log : aggiunge la colonna SalesLog = log(1 + Sales) (solo se
         is_train=True e la colonna Sales e' presente)
    """
    df = df.copy()

    # Variabili temporanee derivate dalla data: servono a piu' trasformazioni
    # (feature cicliche, mesi/settimane di apertura di concorrenza/Promo2,
    # mese per IsPromoMonth), quindi le calcoliamo comunque se la colonna
    # Date e' ancora presente, indipendentemente da quali flag sono attivi.
    if "Date" in df.columns:
        _year = df["Date"].dt.year
        _month = df["Date"].dt.month
        _day_of_year = df["Date"].dt.dayofyear
        _week_of_year = df["Date"].dt.isocalendar().week.astype(int)

    # ------------------------------------------------------------------
    # Giorno dell'anno -> seno/coseno (stagionalita' annuale: dicembre e
    # gennaio sono "vicini" nel tempo, cosa che un singolo intero 1..365 non
    # rappresenta correttamente). Rimuove poi la colonna Date.
    # ------------------------------------------------------------------
    if date_sincos:
        df["DayOfYear_sin"] = np.sin(2 * np.pi * _day_of_year / 365)
        df["DayOfYear_cos"] = np.cos(2 * np.pi * _day_of_year / 365)
        df = df.drop(columns=["Date"])

    # ------------------------------------------------------------------
    # Giorno della settimana -> seno/coseno (stesso ragionamento sul ciclo
    # settimanale). Rimuove poi la colonna DayOfWeek originale.
    # ------------------------------------------------------------------
    if dayofweek_sincos:
        df["DayOfWeek_sin"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
        df["DayOfWeek_cos"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7)
        df = df.drop(columns=["DayOfWeek"])

    # ------------------------------------------------------------------
    # Customers: rimossa perche' non e' nota a tempo di predizione (non e'
    # presente nel test set ufficiale ed e' fortemente correlata a Sales:
    # usarla in training sarebbe una forma di leakage)
    # ------------------------------------------------------------------
    if drop_customers and "Customers" in df.columns:
        df = df.drop(columns=["Customers"])

    # ------------------------------------------------------------------
    # StateHoliday: da stringa ("0","a","b","c") a booleano/intero 0/1
    # 0 -> nessuna festivita', a/b/c -> festivita' (di qualunque tipo)
    # ------------------------------------------------------------------
    if stateholiday_bool:
        df["StateHoliday"] = (df["StateHoliday"] != "0").astype(int)

    # ------------------------------------------------------------------
    # StoreType: one-hot encoding (4 categorie: a, b, c, d). Non essendo una
    # variabile ordinale, il one-hot evita di introdurre un ordinamento
    # arbitrario tra i tipi di negozio, a differenza di un semplice intero.
    # ------------------------------------------------------------------
    if storetype_onehot:
        storetype_dummies = pd.get_dummies(df["StoreType"], prefix="StoreType").astype(int)
        df = pd.concat([df.drop(columns=["StoreType"]), storetype_dummies], axis=1)

    # ------------------------------------------------------------------
    # Assortment: label encoding esplicito e ordinato a=0, b=1, c=2
    # ------------------------------------------------------------------
    if assortment_int:
        df["Assortment"] = df["Assortment"].map({"a": 0, "b": 1, "c": 2})

    # ------------------------------------------------------------------
    # CompetitionDistance: valori mancanti (negozi senza un concorrente
    # mappato) riempiti con la mediana calcolata sul training set
    # ------------------------------------------------------------------
    if fill_competition_distance:
        df["CompetitionDistance"] = df["CompetitionDistance"].fillna(competition_distance_median)

    # ------------------------------------------------------------------
    # CompetitionOpenSinceMonth/Year -> "da quanti mesi e' aperta la
    # concorrenza" rispetto alla data della riga corrente. Mancanti e
    # negativi -> 0 ("nessuna concorrenza attiva al momento").
    # ------------------------------------------------------------------
    if competition_open_months:
        competition_open_months_val = (
            12 * (_year - df["CompetitionOpenSinceYear"])
            + (_month - df["CompetitionOpenSinceMonth"])
        )
        competition_open_months_val = competition_open_months_val.fillna(0).clip(lower=0)
        df["CompetitionOpenMonths"] = competition_open_months_val
        df = df.drop(columns=["CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"])

    # ------------------------------------------------------------------
    # Promo2SinceYear/Week -> stesso trattamento, analogo a quello della
    # concorrenza: "da quante settimane e' attiva Promo2" per quel negozio.
    # Mancanti e negativi -> 0.
    # ------------------------------------------------------------------
    if promo2_open_weeks:
        promo2_open_weeks_val = (
            52 * (_year - df["Promo2SinceYear"])
            + (_week_of_year - df["Promo2SinceWeek"])
        )
        promo2_open_weeks_val = promo2_open_weeks_val.fillna(0).clip(lower=0)
        df["Promo2OpenWeeks"] = promo2_open_weeks_val
        df = df.drop(columns=["Promo2SinceWeek", "Promo2SinceYear"])

    # ------------------------------------------------------------------
    # PromoInterval -> booleano IsPromoMonth: il mese della riga corrente
    # rientra tra quelli in cui riparte un ciclo Promo2 per quel negozio?
    # (PromoInterval elenca i mesi abbreviati in inglese, es. "Jan,Apr,Jul,Oct")
    # ------------------------------------------------------------------
    if promo_interval_bool:
        month_abbr = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
                      7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
        current_month_str = _month.map(month_abbr)
        promo_interval_list = df["PromoInterval"].fillna("").str.split(",")
        df["IsPromoMonth"] = [
            int(m in lst) for m, lst in zip(current_month_str, promo_interval_list)
        ]
        df = df.drop(columns=["PromoInterval"])

    # ------------------------------------------------------------------
    # Store: rimossa se richiesto (l'identificativo negozio non viene usato
    # come feature nel modello)
    # ------------------------------------------------------------------
    if drop_store:
        df = df.drop(columns=["Store"])

    # ------------------------------------------------------------------
    # Target in scala logaritmica: log(1 + Sales). Il test ufficiale non
    # contiene Sales (e' cio' che va predetto), quindi si applica solo a
    # train/validation.
    # ------------------------------------------------------------------
    if sales_log and is_train and "Sales" in df.columns:
        df["SalesLog"] = np.log1p(df["Sales"])

    return df


In [ ]:
print("\n" + "=" * 70)
print("5. APPLICAZIONE DELLA PIPELINE A TRAIN / VALIDATION / TEST")
print("=" * 70)

# Applichiamo la STESSA funzione, con la STESSA mediana (calcolata sul solo
# training set), ai tre dataframe: questo garantisce coerenza tra le feature
# di train, validation e test ed evita data leakage.
train_prep = engineer_features(train_df, competition_distance_median, is_train=True)
val_prep = engineer_features(val_df, competition_distance_median, is_train=True)
test_prep = engineer_features(test_df, competition_distance_median, is_train=False)

print(f"train_prep: {train_prep.shape[0]:,} righe, {train_prep.shape[1]} colonne")
print(f"val_prep:   {val_prep.shape[0]:,} righe, {val_prep.shape[1]} colonne")
print(f"test_prep:  {test_prep.shape[0]:,} righe, {test_prep.shape[1]} colonne")

print("\nColonne train_prep:")
print(list(train_prep.columns))
print("\nColonne test_prep (nessun target, presente Id per la submission):")
print(list(test_prep.columns))


In [ ]:
print("\n" + "=" * 70)
print("6. CONTROLLI DI QUALITA'")
print("=" * 70)

# Nessun valore mancante deve restare dopo la pipeline (requisito per dare
# in input i dati a una rete neurale)
print("Valori mancanti - train:", train_prep.isnull().sum().sum())
print("Valori mancanti - val:  ", val_prep.isnull().sum().sum())
print("Valori mancanti - test: ", test_prep.isnull().sum().sum())

print("\nTipi di dato (train_prep):")
print(train_prep.dtypes)

print("\nAnteprima (train_prep):")
print(train_prep.head())


In [ ]:
print("\n" + "=" * 70)
print("7. DATASET PREPARATI IN MEMORIA")
print("=" * 70)

# I dataset ingegnerizzati vengono ricreati da zero a ogni esecuzione completa del notebook.
print("I dataset restano disponibili nelle variabili:")
print("  - train_prep")
print("  - val_prep")
print("  - test_prep")
print("Nessun file intermedio viene scritto su disco.")

print("\n" + "=" * 70)
print("FEATURE ENGINEERING COMPLETATO")
print("=" * 70)

## Modello LSTM-GRU con covariate future

Questa sezione usa direttamente `train_prep`, `val_prep` e `test_prep`. Ogni esempio appartiene a un singolo negozio: l'LSTM legge le ultime `n_steps_in` righe con `n_features + 1` valori, mentre il GRU riceve il context ripetuto e le covariate note per ciascuno dei prossimi `out_steps` giorni.

In [ ]:
import copy
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

TARGET = "SalesLog"
HISTORY = 28
HORIZON = 7
BATCH_SIZE = 512

# Id e Sales non sono disponibili al momento della previsione; SalesLog e il target.
feature_cols = [
    column for column in test_prep.columns
    if column != "Id" and column in train_prep.columns
]
if TARGET in feature_cols:
    feature_cols.remove(TARGET)
if "Sales" in feature_cols:
    feature_cols.remove("Sales")

print(f"Device: {device}")
print(f"Numero covariate: {len(feature_cols)}")
print(feature_cols)

In [ ]:
# Aggiungiamo le chiavi temporali rimosse dal feature engineering e ordiniamo per negozio/data.
def add_keys(prepared, source):
    result = prepared.reset_index(drop=True).copy()
    source = source.reset_index(drop=True)
    result["Store"] = source["Store"].to_numpy()
    result["Date"] = source["Date"].to_numpy()
    return result.sort_values(["Store", "Date"]).reset_index(drop=True)

train_seq = add_keys(train_prep, train_df)
val_seq = add_keys(val_prep, val_df)
test_seq = add_keys(test_prep, test_df)

feature_scaler = StandardScaler().fit(train_seq[feature_cols])
target_scaler = StandardScaler().fit(train_seq[[TARGET]])

for frame in (train_seq, val_seq, test_seq):
    frame.loc[:, feature_cols] = feature_scaler.transform(frame[feature_cols]).astype(np.float32)
train_seq[TARGET] = target_scaler.transform(train_seq[[TARGET]]).ravel().astype(np.float32)
val_seq[TARGET] = target_scaler.transform(val_seq[[TARGET]]).ravel().astype(np.float32)

print(train_seq.shape, val_seq.shape, test_seq.shape)
print(f"Feature scaling fit su {len(train_seq):,} righe di train")

In [ ]:
def make_windows(frame, target_start, target_end, history=HISTORY, horizon=HORIZON, stride=HORIZON):
    histories, future_features, targets = [], [], []
    for _, store_frame in frame.groupby("Store", sort=False):
        store_frame = store_frame.sort_values("Date").reset_index(drop=True)
        dates = store_frame["Date"].to_numpy()
        first_target = np.searchsorted(dates, np.datetime64(target_start), side="left")
        last_target = np.searchsorted(dates, np.datetime64(target_end), side="left")
        for origin in range(max(history - 1, first_target - 1), last_target - horizon, stride):
            history_slice = store_frame.iloc[origin - history + 1:origin + 1]
            future_slice = store_frame.iloc[origin + 1:origin + horizon + 1]
            if len(history_slice) != history or len(future_slice) != horizon:
                continue
            histories.append(history_slice[feature_cols + [TARGET]].to_numpy(np.float32))
            future_features.append(future_slice[feature_cols].to_numpy(np.float32))
            targets.append(future_slice[TARGET].to_numpy(np.float32)[:, None])
    return tuple(np.asarray(values, dtype=np.float32) for values in (histories, future_features, targets))

train_start, train_end = train_seq["Date"].min(), train_seq["Date"].max() + pd.Timedelta(days=1)
val_start, val_end = val_seq["Date"].min(), val_seq["Date"].max() + pd.Timedelta(days=1)
X_train, C_train, y_train = make_windows(train_seq, train_start, train_end)
X_val, C_val, y_val = make_windows(pd.concat([train_seq, val_seq]), val_start, val_end, stride=1)

# Dataset CPU e pin_memory riducono il costo del trasferimento verso CUDA.
def make_loader(X, C, y, shuffle):
    dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(C), torch.from_numpy(y))
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      pin_memory=torch.cuda.is_available(), num_workers=0)

train_loader = make_loader(X_train, C_train, y_train, shuffle=True)
val_loader = make_loader(X_val, C_val, y_val, shuffle=False)
print(f"Train windows: {len(X_train):,}; validation windows: {len(X_val):,}")
print(f"History: {X_train.shape}; future covariates: {C_train.shape}; target: {y_train.shape}")

In [ ]:
class CovariateLSTMGRU(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.1):
        super().__init__()
        recurrent_dropout = dropout if num_layers > 1 else 0.0
        self.encoder = nn.LSTM(n_features + 1, hidden_size, num_layers=num_layers,
                               batch_first=True, dropout=recurrent_dropout)
        self.decoder = nn.GRU(hidden_size + n_features, hidden_size,
                               num_layers=num_layers, batch_first=True,
                               dropout=recurrent_dropout)
        self.head = nn.Sequential(nn.LayerNorm(hidden_size), nn.Linear(hidden_size, 1))

    def forward(self, history, future_covariates):
        _, (hidden, _) = self.encoder(history)
        context = hidden[-1].unsqueeze(1).expand(-1, future_covariates.size(1), -1)
        decoder_input = torch.cat((context, future_covariates), dim=-1)
        decoded, _ = self.decoder(decoder_input, hidden)
        return self.head(decoded)

model = CovariateLSTMGRU(len(feature_cols)).to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
with torch.inference_mode():
    shape_check = model(torch.zeros(2, HISTORY, len(feature_cols) + 1, device=device),
                        torch.zeros(2, HORIZON, len(feature_cols), device=device))
print(f"Output shape: {tuple(shape_check.shape)}")

In [ ]:
def run_epoch(model, data_loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    criterion = nn.HuberLoss()
    total_loss, total_rows = 0.0, 0
    amp_enabled = device.type == "cuda"
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for history, future_covariates, target in data_loader:
            history = history.to(device, non_blocking=True)
            future_covariates = future_covariates.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                prediction = model(history, future_covariates)
                loss = criterion(prediction, target)
            if training:
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                else:
                    loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
            total_loss += loss.item() * len(history)
            total_rows += len(history)
    return total_loss / total_rows

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5,
                                                       patience=3, min_lr=1e-5)
amp_scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
best_state, best_loss, stale = None, float("inf"), 0
loss_history = {"train": [], "val": []}

for epoch in range(50):
    train_loss = run_epoch(model, train_loader, optimizer, amp_scaler)
    val_loss = run_epoch(model, val_loader)
    scheduler.step(val_loss)
    loss_history["train"].append(train_loss)
    loss_history["val"].append(val_loss)
    if val_loss < best_loss:
        best_loss, best_state, stale = val_loss, copy.deepcopy(model.state_dict()), 0
    else:
        stale += 1
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:02d} | train={train_loss:.4f} | val={val_loss:.4f} | lr={optimizer.param_groups[0]['lr']:.2e}")
    if stale >= 8:
        print(f"Early stopping at epoch {epoch + 1}")
        break

model.load_state_dict(best_state)
plt.figure(figsize=(10, 4))
plt.plot(loss_history["train"], label="train")
plt.plot(loss_history["val"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Huber loss on SalesLog")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def predict_loader(model, data_loader):
    predictions, actuals = [], []
    model.eval()
    with torch.inference_mode():
        for history, future_covariates, target in data_loader:
            predictions.append(model(history.to(device), future_covariates.to(device)).cpu().numpy())
            actuals.append(target.numpy())
    return np.concatenate(predictions), np.concatenate(actuals)

val_pred_scaled, val_true_scaled = predict_loader(model, val_loader)
val_pred = np.expm1(target_scaler.inverse_transform(val_pred_scaled.reshape(-1, 1)))
val_true = np.expm1(target_scaler.inverse_transform(val_true_scaled.reshape(-1, 1)))
print(f"Validation MAE: {np.mean(np.abs(val_pred - val_true)):.2f}")
print(f"Validation RMSE: {np.sqrt(np.mean((val_pred - val_true) ** 2)):.2f}")

## Esempi visuali sulla validation

Qui confrontiamo direttamente le vendite reali con le previsioni del modello. Il primo grafico usa la previsione del primo giorno di ogni finestra; il secondo mostra tutti gli `HORIZON` giorni previsti da una singola finestra.

In [ ]:
def validation_forecast_frame(model, history_frame, validation_frame, store_id, max_days=120):
    past = history_frame[history_frame["Store"] == store_id].sort_values("Date")
    future = validation_frame[validation_frame["Store"] == store_id].sort_values("Date")
    predictions = []
    actuals = []
    dates = []
    model.eval()
    for date in future["Date"].iloc[:max_days]:
        history = past[past["Date"] < date].tail(HISTORY)
        covariate_row = future[future["Date"] >= date].head(HORIZON)
        if len(history) != HISTORY or len(covariate_row) == 0:
            continue
        history_input = history[feature_cols + [TARGET]].to_numpy(np.float32)
        future_input = covariate_row[feature_cols].to_numpy(np.float32)
        with torch.inference_mode():
            prediction = model(
                torch.from_numpy(history_input).unsqueeze(0).to(device),
                torch.from_numpy(future_input).unsqueeze(0).to(device),
            )[0, 0, 0].item()
        predictions.append(prediction)
        actuals.append(future.loc[future["Date"] == date, TARGET].iloc[0])
        dates.append(date)
    result = pd.DataFrame({"Date": dates, "Actual": actuals, "Prediction": predictions})
    result["Actual"] = np.expm1(target_scaler.inverse_transform(result[["Actual"]])).ravel()
    result["Prediction"] = np.expm1(target_scaler.inverse_transform(result[["Prediction"]])).ravel()
    return result

example_store = int(val_seq["Store"].value_counts().index[0])
validation_history = pd.concat([train_seq, val_seq], ignore_index=True)
validation_plot = validation_forecast_frame(model, validation_history, val_seq, example_store)

fig, axes = plt.subplots(2, 1, figsize=(15, 9), constrained_layout=True)
axes[0].plot(validation_plot["Date"], validation_plot["Actual"], label="Sales reali", color="black", linewidth=1.5)
axes[0].plot(validation_plot["Date"], validation_plot["Prediction"], label="Sales previste", color="tab:red", linewidth=1.2)
axes[0].set_title(f"Validation: Store {example_store} - primo giorno previsto per finestra")
axes[0].set_ylabel("Sales")
axes[0].legend()

# Esempio multi-step: una singola finestra di validation, tutti i giorni dell'orizzonte.
example_date = val_seq[val_seq["Store"] == example_store]["Date"].min()
example_history = train_seq[train_seq["Store"] == example_store].sort_values("Date").tail(HISTORY)
example_future = val_seq[(val_seq["Store"] == example_store) & (val_seq["Date"] >= example_date)].sort_values("Date").head(HORIZON)
with torch.inference_mode():
    multi_step_scaled = model(
        torch.from_numpy(example_history[feature_cols + [TARGET]].to_numpy(np.float32)).unsqueeze(0).to(device),
        torch.from_numpy(example_future[feature_cols].to_numpy(np.float32)).unsqueeze(0).to(device),
    ).cpu().numpy()[0, :, 0]
multi_step_prediction = np.expm1(target_scaler.inverse_transform(multi_step_scaled[:, None])).ravel()
axes[1].plot(example_future["Date"], example_future["Sales"], "o-", label="Sales reali", color="black")
axes[1].plot(example_future["Date"], multi_step_prediction, "o--", label="Sales previste", color="tab:blue")
axes[1].set_title(f"Validation multi-step: Store {example_store} da {example_date.date()}")
axes[1].set_ylabel("Sales")
axes[1].legend()
plt.show()

validation_plot.head(HORIZON)

In [ ]:
# Inferenza sul test ufficiale: dopo ogni blocco usiamo le previsioni come target storico.
def forecast_test(model, history_frame, future_frame):
    rows = []
    model.eval()
    for store_id, future_store in future_frame.groupby("Store", sort=False):
        past_store = history_frame[history_frame["Store"] == store_id].sort_values("Date")
        future_store = future_store.sort_values("Date").reset_index(drop=True)
        history_features = past_store[feature_cols].to_numpy(np.float32)
        history_target = past_store[TARGET].to_numpy(np.float32)
        for start in range(0, len(future_store), HORIZON):
            current = future_store.iloc[start:start + HORIZON]
            future_covariates = current[feature_cols].to_numpy(np.float32)
            history_input = np.column_stack((history_features[-HISTORY:], history_target[-HISTORY:]))
            with torch.inference_mode():
                prediction = model(
                    torch.from_numpy(history_input).unsqueeze(0).to(device),
                    torch.from_numpy(future_covariates).unsqueeze(0).to(device),
                ).squeeze(0).squeeze(-1).cpu().numpy()
            rows.append(pd.DataFrame({
                "Id": current["Id"].to_numpy(),
                "Store": store_id,
                "Date": current["Date"].to_numpy(),
                "SalesLogPrediction": prediction,
            }))
            history_features = np.vstack((history_features, future_covariates))
            history_target = np.concatenate((history_target, prediction))
    return pd.concat(rows, ignore_index=True)

# Per il test ufficiale la storia include anche la validation, poiche Sales e nota fino al cutoff.
history_for_test = pd.concat([train_seq, val_seq], ignore_index=True)
test_predictions = forecast_test(model, history_for_test, test_seq)
test_predictions["Sales"] = np.maximum(
    0.0, np.expm1(target_scaler.inverse_transform(test_predictions[["SalesLogPrediction"]]))[:, 0]
)
test_predictions = test_predictions.sort_values("Id")
submission = test_predictions[["Id", "Sales"]]
submission.to_csv(os.path.join(OUTPUT_DIR, "submission_lstm_gru_covariates.csv"), index=False)
print(submission.head())
print(f"Submission salvata: {len(submission):,} righe in {OUTPUT_DIR}/submission_lstm_gru_covariates.csv")